# Carleton evacuation — DEVS

Set `SCENARIO` (1–7) and `RUN_SIMULATION` below, then run all cells.

Same layout as the MARS notebook: pick scenario → optional sim → charts.

| # | Scenario |
|---|----------|
| 1 | Baseline |
| 2 | P6 +1 h |
| 3 | P6 +2 h |
| 4 | P3 +1 h, P6 +2 h |
| 5 | P3/P4 +1 h, P6 +2 h |
| 6 | P3/P4 +1 h, P6 +1.5 h |
| 7 | Baseline timing; P3/P4 emergency exit (r28) |

Run this notebook from the DEVS repo root (or WSL), with `bin/campus_evacuation` already built.

**Step 1** — Load the project folder and helpers for running commands.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "main" / "main.cpp").is_file():
    for candidate in (ROOT / "model-campus-evacuation-main", ROOT.parent):
        if (candidate / "main" / "main.cpp").is_file():
            ROOT = candidate
            break

os.chdir(ROOT)

BIN = ROOT / "bin" / "campus_evacuation"
RAW = ROOT / "output_data" / "raw"
PROCESSED = ROOT / "output_data" / "processed"
SCHEDULES = ROOT / "input_data" / "parking_lot_schedules"
SCENARIOS = ROOT / "input_data" / "scenarios"


def run_cmd(cmd: list[str], *, quiet: bool = False, label: str | None = None) -> None:
    if label:
        print(label, flush=True)
    elif not quiet:
        print("Running:", " ".join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=ROOT, capture_output=quiet, text=True)
    if completed.returncode != 0:
        details = ""
        if quiet:
            details = "\n" + (completed.stdout or "") + (completed.stderr or "")
        raise RuntimeError(
            f"Command failed ({completed.returncode}): {' '.join(cmd)}{details}"
        )
    if quiet:
        print("  done.", flush=True)


assert (ROOT / "main" / "main.cpp").is_file(), f"Open this notebook from the DEVS repo root ({ROOT})"
print(f"Ready. Project folder:\n  {ROOT}")

**Step 2** — Pick the scenario and whether to run a new simulation or reuse existing results.

In [ ]:
SCENARIO = 1
RUN_SIMULATION = False
BUILD_FIRST = False

SID = f"{SCENARIO:02d}"
NAME = f"scenario_{SID}"
LOG_PATH = RAW / f"{NAME}_log.csv"
SCHEDULE = SCHEDULES / f"{NAME}.csv"
MANIFEST = SCENARIOS / f"{NAME}.json"  # scenario_07 only today

assert 1 <= SCENARIO <= 7, f"Choose scenario 1–7 (got {SCENARIO})"
if SCENARIO != 7:
    assert SCHEDULE.is_file(), f"Missing schedule: {SCHEDULE}"
else:
    assert MANIFEST.is_file(), f"Missing manifest: {MANIFEST}"

print(f"{NAME}  |  run={RUN_SIMULATION}  |  log: {LOG_PATH.relative_to(ROOT)}")

**Step 3** — Build (optional) and run the DEVS / Cadmium simulation (skipped when `RUN_SIMULATION` is False).

Requires Cadmium installed and `bin/campus_evacuation` available (or `BUILD_FIRST = True`).

In [ ]:
if RUN_SIMULATION:
    if BUILD_FIRST:
        run_cmd(["bash", "build_sim.sh"], label="Building…")
    if not BIN.is_file():
        raise FileNotFoundError(
            f"Missing binary: {BIN}\nBuild first: bash build_sim.sh  (or set BUILD_FIRST = True)"
        )

    RAW.mkdir(parents=True, exist_ok=True)
    cmd = [str(BIN)]
    if SCENARIO == 7:
        cmd += ["--scenario", "scenario_07", "-o", str(LOG_PATH.relative_to(ROOT)).replace("\\", "/")]
    else:
        cmd += [
            "-i",
            str(SCHEDULE.relative_to(ROOT)).replace("\\", "/"),
            "-o",
            str(LOG_PATH.relative_to(ROOT)).replace("\\", "/"),
        ]

    run_cmd(cmd, label="Running simulation…")
    if not LOG_PATH.is_file():
        raise FileNotFoundError(f"Simulation finished but log missing: {LOG_PATH}")
    stat = LOG_PATH.stat()
    mtime = datetime.fromtimestamp(stat.st_mtime)
    print(
        f"Log: {LOG_PATH.name}  |  {stat.st_size:,} bytes  |  "
        f"modified {mtime:%Y-%m-%d %H:%M:%S}",
        flush=True,
    )
    print(f"Finished. Log in: {LOG_PATH.relative_to(ROOT)}", flush=True)
else:
    print("Simulation skipped.")

**Step 4** — Analyze the log and show summary, evacuation curve, and heatmap.

Uses `analysis/data_analysis.py` + `analysis/visualize_processed.py` (same pipeline as `run_scenarios.sh`).

In [ ]:
if not LOG_PATH.is_file():
    raise FileNotFoundError(
        f"No log at {LOG_PATH}. Set RUN_SIMULATION = True and re-run Step 3."
    )

PROCESSED.mkdir(parents=True, exist_ok=True)
py = sys.executable
analyze = [
    py,
    str(ROOT / "analysis" / "data_analysis.py"),
    str(LOG_PATH),
]
if SCENARIO == 7:
    analyze += ["--scenario-manifest", str(MANIFEST)]

run_cmd(analyze)
run_cmd([py, str(ROOT / "analysis" / "visualize_processed.py"), str(PROCESSED)])

# Keep a per-scenario copy of the heatmap (shared processed/ is overwritten each run)
heat_csv = PROCESSED / "heatmap_matrix.csv"
heat_png = PROCESSED / "heatmap_matrix.png"
if heat_csv.is_file():
    shutil.copy2(heat_csv, PROCESSED / f"{NAME}_heatmap_matrix.csv")
if heat_png.is_file():
    shutil.copy2(heat_png, PROCESSED / f"{NAME}_heatmap.png")

summary_csv = PROCESSED / "summary.csv"
if summary_csv.is_file():
    display(Markdown("### Summary"))
    print(summary_csv.read_text(encoding="utf-8"))

for name, title in {
    "evac_curve.png": "### Evacuation curve",
    "heatmap_matrix.png": "### Heatmap",
}.items():
    path = PROCESSED / name
    if path.is_file():
        display(Markdown(title))
        display(Image(filename=str(path)))
    else:
        print(f"Missing chart: {name}")